# Notebook 1 — corda contra barra

Minicurso **Da kalimba à viga** · Bloco 1

Duas tarefas: (i) sintetizar duas notas, uma com razões harmônicas e outra com as razões da
barra, e ouvir a diferença; (ii) olhar o espectro de uma gravação da kalimba e da corda.

## 1. Sintetizar as duas séries

In [ ]:
import numpy as np
from IPython.display import Audio, display

fs = 44100                        # taxa de amostragem [Hz]
t  = np.arange(0, 2.0, 1/fs)      # 2 segundos
f1 = 440.0                        # fundamental [Hz]

razoes_corda = np.array([1, 2, 3, 4, 5])
razoes_barra = np.array([1, 6.267, 17.55, 34.39])   # viga engastada-livre

def sintetiza(f1, razoes, decaimento=3.0):
    """Soma senos nas razoes dadas, com os modos altos mais fracos."""
    som = np.zeros_like(t)
    for i, r in enumerate(razoes):
        amp = 1.0 / (i + 1)
        som += amp * np.sin(2*np.pi*f1*r*t) * np.exp(-decaimento*t)
    return som / np.max(np.abs(som))

print("a corda:")
display(Audio(sintetiza(f1, razoes_corda), rate=fs))
print("a barra:")
display(Audio(sintetiza(f1, razoes_barra), rate=fs))

**Experimente.** Na kalimba real os modos altos somem mais depressa que o
fundamental. Dê um decaimento diferente a cada modo e veja como a síntese fica mais
convincente.

In [ ]:
def sintetiza2(f1, razoes, decaimentos):
    som = np.zeros_like(t)
    for i, (r, d) in enumerate(zip(razoes, decaimentos)):
        som += (1.0/(i+1)) * np.sin(2*np.pi*f1*r*t) * np.exp(-d*t)
    return som / np.max(np.abs(som))

# os modos altos decaem mais rapido
dec = [2.0, 8.0, 14.0, 20.0]
Audio(sintetiza2(f1, razoes_barra, dec), rate=fs)

## 2. O espectro de uma gravação

Grave a kalimba e a corda com o celular, passe os arquivos para `.wav` e rode a célula
abaixo. Grave pelo menos um segundo: a resolução em frequência é o inverso da duração.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import wavfile

def espectro(arquivo, fmax=5000):
    fs, dados = wavfile.read(arquivo)
    if dados.ndim > 1:
        dados = dados[:, 0]                 # fica com um canal so
    dados = dados.astype(float)
    dados /= np.max(np.abs(dados))

    amp = np.abs(np.fft.rfft(dados))
    freqs = np.fft.rfftfreq(len(dados), 1/fs)

    plt.figure(figsize=(9, 4))
    plt.plot(freqs, amp, lw=0.8)
    plt.xlim(0, fmax)
    plt.xlabel("frequencia [Hz]"); plt.ylabel("amplitude")
    plt.title(arquivo)
    plt.tight_layout(); plt.show()
    return freqs, amp

# freqs, amp = espectro("kalimba.wav")

### Achar os picos

A função abaixo devolve os picos mais altos, que são os modos. Compare as razões entre
eles com 1 : 2 : 3 (corda) e 1 : 6,27 : 17,55 (barra).

In [ ]:
from scipy.signal import find_peaks

def modos(freqs, amp, quantos=4, altura=0.08):
    idx, _ = find_peaks(amp, height=altura*np.max(amp), distance=50)
    idx = idx[np.argsort(amp[idx])[::-1][:quantos]]
    f = np.sort(freqs[idx])
    print("picos [Hz] :", np.round(f, 1))
    print("razoes     :", np.round(f/f[0], 3))
    return f

# modos(freqs, amp)